In [23]:

import numpy as np
from math import sqrt
import warnings

import pandas as pd
from pandas.core.common import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)

In [24]:
data = pd.read_csv("data/symbol_pins.csv")
data.head()

,Symbol Name,Pin Name,Pin CenterY,Pin CenterX,Pin Width,Pin Height
0,90980-12A87,8,-5759.0,10368.0,0.0,0.0
1,90980-12A87,10,-13439.0,10368.0,0.0,0.0
2,90980-12A87,23,-17279.0,5760.0,0.0,0.0
3,90980-12A87,6,1920.0,10368.0,0.0,0.0
4,90980-12A87,15,13440.0,5760.0,0.0,0.0


In [25]:
len(data)

7341

In [26]:
data.isnull().sum()

Symbol Name    0
Pin Name       0
Pin CenterY    0
Pin CenterX    0
Pin Width      0
Pin Height     0
dtype: int64

In [27]:
def cal_distance_center(row):
    xs= 0-float(row["Pin CenterX"])
    ys= 0-float(row["Pin CenterY"])
    return sqrt((xs*xs)+(ys*ys))

In [28]:
data["distance_to_center"]=data.apply(lambda row: cal_distance_center(row), axis=1)

In [29]:
cols=['Symbol Name', 'Pin Name', 'Pin CenterY', 'Pin CenterX', 'Pin Width',
       'Pin Height', 'distance_to_center', 'neighbors']

dataout=pd.DataFrame([],columns=cols)

In [30]:
list_pn=list(data['Symbol Name'].unique())
#list_pn

In [31]:
def get_neighbors(row, distances):
    pin_row=row["Pin Name"]
    distances_sorted=sorted(distances[pin_row], key=distances[pin_row].get, reverse=False)
    ts=distances[pin_row][distances_sorted[0]]*1.9
    return [x for x in  distances_sorted if distances[pin_row][x] < ts]

In [32]:
for pn in list_pn:
    data1 = data[data['Symbol Name']==pn]
    if len(data1) > 1:
        
        data_json= data1.to_json(orient="records")
        dicc_json1=eval(data_json)
        dicc_json2=eval(data_json)
        distances={}
        for ele in dicc_json1:
            pin=ele['Pin Name']
            distances[pin]={}
            x1=ele['Pin CenterX']
            y1=ele['Pin CenterY']
            for ele2 in dicc_json2:
                pin2=ele2['Pin Name']
                if ele2 != ele:

                    x2=ele2['Pin CenterX']
                    y2=ele2['Pin CenterY'] 
                    distance=sqrt(((x1-x2)*(x1-x2))+((y1-y2)*(y1-y2)))
                    distances[pin][pin2]=distance
        #  get neighbors
        data1['neighbors']= data1.apply(lambda row: get_neighbors(row, distances), axis=1)
        
        dataout= pd.concat([dataout, data1])
    else:
        #print(pn)
        data1['neighbors']=""
        
        dataout= pd.concat([dataout, data1])

In [33]:
dataout.isnull().sum()

Symbol Name           0
Pin Name              0
Pin CenterY           0
Pin CenterX           0
Pin Width             0
Pin Height            0
distance_to_center    0
neighbors             0
dtype: int64

In [34]:
dataout["internal_pin"]=""

In [35]:
cols=['Symbol Name', 'Pin Name', 'Pin CenterY', 'Pin CenterX', 'Pin Width',
       'Pin Height', 'distance_to_center', 'neighbors','internal_pin']

dataout2=pd.DataFrame([],columns=cols)

In [36]:
# def internal_external_pin(row, ndistances, xg,xy, distance_to_center):

#     nave= np.mean(ndistances)
#     xg=0-(np.sum(xs)/len(xs))
#     yg=0-(np.sum(ys)/len(ys))
#     centroide_distance=sqrt((xg*xg)+(yg*yg))
#     print(f"{centroide_distance} {distance_to_center}")
#     return (centroide_distance*1.1> distance_to_center>centroide_distance*0.90) and (distance_to_center>centroide_distance)

In [37]:
for pn in list_pn:
    data1 = dataout[dataout['Symbol Name']==pn]
    if len(data1) > 4:
        for i, row in data1.iterrows():
            data_json= data1.to_json(orient="records",force_ascii=False)
            #dicc_json1=eval(data_json)
            dicc_json2=eval(data_json)
            ndistances=[]
            xs=[]
            ys=[]

            pin=data1.at[i,'Pin Name']

            distance_to_center=data1.at[i,'distance_to_center']
            neighbors=data1.at[i,"neighbors"]

            for ele2 in dicc_json2:
                pin2=ele2['Pin Name']

                if pin2 in neighbors:
                    ndistances.append(ele2['distance_to_center'])
                    xs.append(ele2['Pin CenterX'])
                    ys.append(ele2['Pin CenterY'])


                ndistances=sorted(ndistances)
            nave= np.mean(ndistances)
            xg=0-(np.sum(xs)/len(xs))
            yg=0-(np.sum(ys)/len(ys))
            centroide_distance=sqrt((xg*xg)+(yg*yg))
            print(f"{centroide_distance} {distance_to_center}")
            data1.at[i,'internal_pin']= str((centroide_distance*1.1> distance_to_center>centroide_distance*0.90) 
                                            and (distance_to_center>centroide_distance))

        dataout2 = pd.concat([dataout2, data1])
    else:
        #print(pn)
        data1['internal_pin']=str(False)
        dataout2 = pd.concat([dataout2, data1])


9538.067479316762 11860.080311701098
15440.705011106196 16973.571957605152
19267.4870193287 18213.77064201699
7841.975924472096 10544.279207228914
15915.584030754258 14622.284363258705
10287.77410521829 8145.1630431809035
8738.429561425783 6071.573107523288
15914.739584422989 14621.365223535044
12837.9521747045 11194.570157000224
10288.333929261822 8145.870119269028
12246.16879844468 14129.947770604107
20554.26534809746 21891.36816190345
19942.54788135157 23527.63957561404
8738.209850993508 6071.256953877014
7841.7310971494035 10544.09716381635
18877.83068151635 20150.912262227732
9538.671303698435 11860.565922417025
20553.36237699321 21890.403399663515
18878.74599225277 20151.769748585357
15441.57538076993 16974.363728870663
19268.383820133953 18214.719322569865
12838.699896796405 11195.427638102978
19941.617211249442 23526.741912130543
12245.3848955433 14129.268381625427
8164.547774439875 13223.891144440051
8525.067012053336 7625.961513147047
10037.362215741743 12341.635588527155
867

16875.15846684854 16847.265416084592
33509.26845250762 35695.808073217784
27990.031516818624 27966.01998855039
32400.644275144354 33279.4906962231
29836.291170003857 30284.706982237753
48120.30038254385 48129.20618501826
40491.84161076463 40476.77539033958
26836.862447014926 25397.9939759029
19789.125510065496 19766.87375383371
42493.07227452641 42498.542186762126
34628.97596233536 37566.933571959264
10227.249202498197 10226.967683531615
8452.00013310459 8452.000059157595
10247.670271822763 10228.657047726256
10226.967683531615 14286.379107387567
10228.657047726256 14316.055497238058
12519.608320550607 14264.509385183916
12188.005677714464 14166.425660695078
12556.384623469448 14263.634634972952
12371.115400399432 17514.125270763598
9600.000013020834 11904.0
10732.790902766157 12507.723094152669
12216.317942817304 14165.883558747757
12370.475263707534 17649.385768348995
10733.126291998991 12508.03006072499
11774.476827549493 8244.363953635235
11774.782545762788 8244.829652576213
12923.

7142.812891291497 10488.192217918206
7142.812891291497 7142.812891291497
4639.6827747163925 4639.4759402329055
4640.096577658702 4639.889653860316
7142.006510778326 10487.276910618886
7142.006510778326 7142.006510778326
12598.088267669822 17996.60823599825
12883.776503805086 9913.714994894699
11863.285993349398 14978.923459314425
12599.9168648051 17998.52841206747
11864.548242558585 14980.076935716987
14170.329036405612 14065.065410441573
11524.000085039916 8068.000061973227
14168.703116375895 14062.608150695232
10372.00009448515 13828.000036158519
12882.614119812795 9911.971953148375
5856.120606302056 10623.238912873983
5854.998586203455 10614.933348825136
7008.600824701034 3781.000132240146
5933.000569694899 9159.001364777712
7579.498502906084 6574.194399316162
7582.11598287561 6573.376681736716
11882.980745587362 10497.05492030979
9076.58360838482 7147.667171882026
9081.77443454747 7169.054679663142
8376.723087222115 10593.234916681495
11355.34651871091 13090.477034852473
7278.07369

42173.340856991636 41447.103638251974
41911.43742750897 42749.140634637326
38231.718078579725 39148.240126473116
34586.863795955825 35597.35362354904
26856.55646206341 23198.537906514714
29573.609857438776 30743.990632317073
14496.247238509697 13576.450198781713
45617.58550427675 46388.40959765704
42172.3848038026 41446.13083268449
27853.90385637173 26741.572148996776
45857.448250420566 45190.45564054428
21780.790068314785 20442.365420860668
49566.039583973215 48949.60675020791
37825.94918835481 36979.81806337073
12512.888371595105 9789.921399071598
46598.4276884961 45942.1939397761
13640.174487153747 11194.913175188096
12811.105582267286 15331.172949256035
11603.056605912083 14337.008927945884
49344.207636155224 50057.6793808902
21780.790068314785 22969.492114541845
38517.60272966115 37721.047718747155
53086.99226025147 53750.80636604441
13174.676314809409 17147.22321543637
13640.596725950078 11195.427638102978
12810.656003499587 15330.797272157766
54609.59389154986 56500.540183258425

8145.870119269028 8145.870119269028
6071.415012169733 6071.256953877014
8145.1630431809035 11194.570157000224
6071.731239934785 6071.573107523288
8145.1630431809035 8145.1630431809035
8145.870119269028 11195.427638102978
16586.165503816726 14300.895076882425
18227.69761775865 19271.81413359936
11332.003573949312 8452.015144330966
13430.439168173169 10237.112874243401
12227.208716628666 15334.128863420967
16967.30880251786 22372.085910795176
15752.733635785251 18293.31211126077
12242.067393316376 15335.255622258144
18208.20565020068 19221.01644034467
16562.807974495146 14274.28852867981
15780.385962326776 18295.201146748837
16995.004576508814 22402.377039055475
13420.920592958591 10218.529884479469
28593.872862031123 32955.087634536794
16264.413795495708 14963.857925013857
30743.507717529206 29730.312628695985
27949.773790140054 28597.475692794986
31268.71541972903 33521.57706612265
20532.525187187766 21435.500390707002
8413.869457627685 8421.047975163186
34232.33524388893 36590.1252389

...

105684.9012957333 104088.79409427318
55659.06227956055 55508.92339074863
15238.823334660718 17051.741611929265
40093.26736264457 40807.303672259455
22259.12271406939 19584.0
60231.44693596527 62308.659301898
23662.11256476855 23658.887885951022
62512.43718753093 62966.122018113834
59522.20210980101 60229.270757664
21092.43144027023 22430.47435967416
22632.48497182755 20664.76653630522
95503.60248062781 96441.35345379595
33496.020899669726 35800.66412791808
36431.41517375492 37225.23032836735
38255.06403896091 37626.12198991546
86867.02812920447 89395.3319586655
70072.97233286526 70487.47302180722
17996.806912963755 19544.434706585915
85226.70549448948 84592.50690220736
80930.05653512035 79257.63720929359
106840.21179575834 109088.44587764554
28788.388650827805 28873.667328553885
12320.623360853135 9223.996530788592
24213.980006966594 25574.16909696188
12831.361580128587 18753.20132670686
32261.546245026755 32418.224889712885
32299.098399869374 32440.61454411738
56859.236716649655 56437

35670.6238810383 36309.83470080799
41275.92770686687 40845.038670565606
6804.49733632103 6804.49733632103
13476.19645597377 12274.086564791696
13021.230295175643 14545.74869162808
16523.527722614203 17715.339398385797
7190.162579524888 9637.013334015886
6994.801646937532 4244.000117813382
13479.879239815169 12275.964361303759
7977.409002928206 5715.340759744777
7189.095319996807 9611.214283325495
17609.13899403123 21145.886715860368
9787.741099967858 11718.118108297083
10394.405081581148 8772.868402067821
6076.601895796696 8835.000056593095
18166.206948311716 19660.341451765278
17605.175608957223 21136.121356578173
16517.47354167701 17713.605900550006
9799.364309994808 11705.080478151356
13019.461156284464 14532.580362757331
16878.248114066817 15923.731786236542
16884.795444422773 15925.66011190745
18170.583568565493 19666.38617031609
10387.55880657241 8761.940481423051
7983.530291794477 5716.680505328245
6372.501417222283 9268.004369873808
7850.2151324024235 4411.434460580821
5407.333

26284.154410626044 40925.19734833297
26051.146659024434 30134.87099690324
26441.35706199665 29959.098551191422
26697.691925312363 34884.5531718553
14904.19261745248 14538.787879324742
26215.389388022162 26021.77503553514
33691.829837514015 37970.48792154244
24706.662004406826 23163.2679257483
35246.47264088989 38856.86400110024
22219.00865125275 23545.657710074698
15809.186187348481 15395.748796339853
29111.649012723414 32253.561121215746
17385.673506232255 18306.20992996639
21974.200349856306 21669.924434570603
13345.447878583918 10117.454274668109
12049.90315811708 8404.093347887088
25471.548552846172 26380.73624825509
20146.8138225378 18149.450928333892
23856.500883210534 24240.327658676564
32181.910408730484 35464.52967402782
16086.228507639693 13590.55885532306
15981.0363644305 15575.462272433522
33998.121176479915 34642.136842290776
20507.208153232365 18561.631501567957
35223.75805564335 41278.30153967094
16415.32657731792 13946.27785468223
21622.601316536307 22034.402646770344
3

16787.887777799802 14777.349187185095
16039.626480688383 18471.270692618848
18497.53629540972 19793.390437214137
17355.678753652937 22684.710643955765
12892.931088003224 10137.284892908949
16039.626480688383 18471.270692618848
10412.728750908667 6705.355546128781
12892.931088003224 10137.284892908949
10412.728750908667 6705.355546128781
9157.584923985145 12952.975912893531
17355.678753652937 22684.710643955765
16787.887777799802 14777.349187185095
18497.53629540972 19793.390437214137
11902.273465183029 15019.398689694604
9157.584923985145 12952.975912893531
11902.273465183029 15019.398689694604
6071.731239934785 6071.573107523288
8145.1630431809035 8145.1630431809035
6071.415012169733 6071.256953877014
8145.1630431809035 11194.570157000224
8145.870119269028 8145.870119269028
8145.870119269028 11195.427638102978
21456.192142363005 25258.02021141008
10368.00090556409 11904.000672042992
12918.192307749563 14168.59456685807
14516.770233078707 16568.322787777888
15304.31593538241 13631.8319

12695.251395699102 15911.692304717308
12695.251395699102 15911.692304717308
12205.598715343709 11371.847343329931
12205.598715343709 11371.847343329931
13440.0 6144.0
9603.000013016766 3825.1887535126943
9002.466945849405 11787.298927235195
9002.559345973665 11787.087214405432
8659.545084038768 8081.126901614651
8660.967851516365 8080.818089772842
13356.404315907781 12502.134217804574
13845.335391788496 17052.232258563687
13845.279933769647 17052.36734298203
13356.436652883882 12501.949967905006
14592.000008566338 7296.0
16088.277696509344 14306.396122014796
19410.66889339182 23213.745970868207
12856.706914291855 15179.63085848928
18502.98021400877 20156.767424366437
20238.95953627832 20990.447541679525
16080.536952477676 14280.251853521351
13616.6055050442 11401.064730980173
11870.978476941147 9294.517308607263
11859.554839874892 9319.884655938613
10306.602802087602 13111.003089008866
19403.183075063855 23251.22175284559
19025.595472415574 17515.22155155338
12858.358602869965 15100.18

In [16]:
dataout2[dataout2['Symbol Name']=='82675-60070'].head()

,Symbol Name,Pin Name,Pin CenterY,Pin CenterX,Pin Width,Pin Height,distance_to_center,neighbors,internal_pin
7299,82675-60070,1,3990.0,10153.0,0.0,0.0,10908.872948,NaN,False


In [17]:
len(dataout2),len(dataout)

(7341, 7341)

In [18]:
dataout2.head(10)

,Symbol Name,Pin Name,Pin CenterY,Pin CenterX,Pin Width,Pin Height,distance_to_center,neighbors,internal_pin
0,90980-12A87,8,-5759.0,10368.0,0.0,0.0,11860.080312,"[7, 9, 20, 21, 19]",False
1,90980-12A87,10,-13439.0,10368.0,0.0,0.0,16973.571958,"[11, 9, 22, 23, 21]",True
2,90980-12A87,23,-17279.0,5760.0,0.0,0.0,18213.770642,"[22, 24, 11, 10, 12]",False
3,90980-12A87,6,1920.0,10368.0,0.0,0.0,10544.279207,"[7, 5, 18, 19, 17]",False
4,90980-12A87,15,13440.0,5760.0,0.0,0.0,14622.284363,"[14, 16, 3, 4, 2]",False
5,90980-12A87,20,-5759.0,5760.0,0.0,0.0,8145.163043,"[21, 19, 8, 7, 9]",False
6,90980-12A87,18,1920.0,5760.0,0.0,0.0,6071.573108,"[19, 17, 6, 7, 5]",False
7,90980-12A87,22,-13439.0,5760.0,0.0,0.0,14621.365224,"[23, 21, 10, 11, 9]",False
8,90980-12A87,21,-9599.0,5760.0,0.0,0.0,11194.570157,"[20, 22, 9, 8, 10]",False
9,90980-12A87,17,5760.0,5760.0,0.0,0.0,8145.870119,"[18, 16, 5, 6, 4]",False


In [20]:
dataout2.to_pickle('data/symbols_distances.pkl')

In [39]:
data1 = dataout2[dataout2['Symbol Name']=="90980-12745"] #90980-12379 #90980-12745
data1

,Symbol Name,Pin Name,Pin CenterY,Pin CenterX,Pin Width,Pin Height,distance_to_center,neighbors,internal_pin
5406,90980-12745,2,-8637.0,14587.0,0.0,0.0,16952.236962,"[1, 3, 8, 9, 7, 14, 15]",False
5407,90980-12745,8,-9597.0,8832.0,0.0,0.0,13042.493358,"[9, 14, 7, 2, 15, 13]",True
5408,90980-12745,1,-14391.0,14565.0,0.0,0.0,20475.353623,"[7, 2, 8]",False
5409,90980-12745,17,9590.0,4224.0,0.0,0.0,10479.039842,"[16, 11, 18, 10, 12]",False
5410,90980-12745,14,-9593.0,4224.0,0.0,0.0,10481.785392,"[15, 8, 13, 9, 7]",False
5411,90980-12745,15,-5753.0,4224.0,0.0,0.0,7137.169257,"[14, 9, 8]",False
5412,90980-12745,4,2875.0,14634.0,0.0,0.0,14913.737996,"[3, 5, 10, 11, 9, 16]",False
5413,90980-12745,6,14420.0,14596.0,0.0,0.0,20517.787795,"[12, 5, 11]",False
5414,90980-12745,10,5750.0,8832.0,0.0,0.0,10538.819858,"[11, 16, 17, 4, 5]",False
5415,90980-12745,11,9590.0,8874.0,0.0,0.0,13065.832388,"[10, 17, 12, 5, 16, 18]",True


In [ ]:
data_json= data1.to_json(orient="records")
dicc_json1=eval(data_json)
dicc_json2=eval(data_json)

In [ ]:
ndistances=[]
xs=[]
ys=[]
for ele in dicc_json1:
    pin=ele['Pin Name']
    print(pin)
    distance_to_center=ele['distance_to_center']
    neighbors=ele["neighbors"]
    xs=[]
    ys=[]
    for ele2 in dicc_json2:
        pin2=ele2['Pin Name']
        
        if pin2 in neighbors:
            ndistances.append(ele2['distance_to_center'])
            xs.append(ele2['Pin CenterX'])
            ys.append(ele2['Pin CenterY'])
            
            
        ndistances=sorted(ndistances)

    nave= np.mean(ndistances)
    xg=0-(np.sum(xs)/len(xs))
    yg=0-(np.sum(ys)/len(ys))
    centroide_distance=sqrt((xg*xg)+(yg*yg))
    print(f"{centroide_distance} {distance_to_center}")
    print((centroide_distance*1.1> distance_to_center>centroide_distance*0.90) and (distance_to_center>centroide_distance))
    #print(nave*1.05> distance_to_center>nave*0.95)
    ndistances=[]


In [ ]:
print(centroide_distance*1.1> distance_to_center>centroide_distance*0.90)

In [ ]:
distances={}
for ele in dicc_json1:
    pin=ele['Pin Name']
    distances[pin]={}
    x1=ele['Pin CenterX']
    y1=ele['Pin CenterY']
    for ele2 in dicc_json2:
        pin2=ele2['Pin Name']
        if ele2 != ele:

            x2=ele2['Pin CenterX']
            y2=ele2['Pin CenterY'] 
            distance=sqrt(((x1-x2)*(x1-x2))+((y1-y2)*(y1-y2)))
            distances[pin][pin2]=distance


In [ ]:
 data1['neighbors']= data1.apply(lambda row: get_neighbors(row, distances), axis=1)

In [ ]:
data1

In [ ]:
distances['2']

In [ ]:
distances_sorted=sorted(distances['2'], key=distances['2'].get, reverse=False)
distances_sorted

In [ ]:
distances["2"][distances_sorted[0]]*1.9

In [ ]:
distances["2"][distances_sorted[0]]

In [ ]:
distances_sorted=sorted(distances['2'], key=distances['2'].get, reverse=False)
ts=distances["2"][distances_sorted[0]]*2
ts

In [ ]:
[x for x in  distances_sorted if distances["2"][x] < ts]

In [ ]:
def get_neighbors(row, distances):
    pin_row=row["Pin Name"]
    distances_sorted=sorted(distances[pin_row], key=distances[pin_row].get, reverse=False)
    ts=distances[pin_row][distances_sorted[0]]*1.95
    return [x for x in  distances_sorted if distances[pin_row][x] < ts]
    

In [ ]:
data1['neighbors']= data1.apply(lambda row: get_neighbors(row, distances), axis=1)

In [ ]:
data1.columns

In [ ]:
data1[["Pin Name",'neighbors']]

In [ ]:
a=17114.971721
b=17110.687304

c= sqrt((a*a)+(b*b))
c

In [ ]:
x1=16896.0 # (pin 2 and 3)
x2=16896.0 
y1=2729.0
y2=-2702.0

distance=sqrt((x1-x2)**2+(y1-y2)**2)
distance

In [ ]:
x1=16896.0 # (pin 2 and 1)
x2=16896.0 
y1=2729.0
y2=8160.0

distance=sqrt((x1-x2)**2+(y1-y2)**2)
distance

In [ ]:
x1=16896.0 # (pin 2 and 6)
x2=11466.0	 
y1=2729.0
y2=2729.0	

distance=sqrt((x1-x2)**2+(y1-y2)**2)
distance

In [ ]:
x1=16896.0 # (pin 2 and 5)
x2=11466.0 
y1=2729.0
y2=-2702.0

distance=sqrt((x1-x2)**2+(y1-y2)**2)
distance

In [ ]:
x1=16896.0 # (pin 2 and 5)
x2=11466.0 
y1=2729.0
y2=8160.0

distance=sqrt((x1-x2)**2+(y1-y2)**2)
distance

In [ ]:
data["Signal Name"].fillna("na", inplace=True)
data["Pin PreferredSignal"].fillna("na", inplace=True)

In [ ]:
def signal_equal(row):
    if (row["Pin PreferredSignal"]== "na" and row["Signal Name"]== "na") :
        return 2
    elif row["Pin PreferredSignal"]==row["Signal Name"]:
        return 1
    else:
        return 0

In [ ]:
data['check_signal']=data.apply(lambda row:signal_equal(row),axis=1)

In [ ]:
connector_view=data.groupby(["Connector Name",'Pin Name']).agg({ "Pin PreferredSignal":['nunique','count'] })
connector_view.head(10)

In [ ]:
levels=connector_view.index.nlevels
for i in reversed(range(0,levels)):
    connector_view.reset_index(level=i, inplace=True)

In [ ]:
connector_view.columns=['Connector Name','Pin Name','Pin Name nunique',"Pin Name count"]
connector_view.tail(10)

In [ ]:
groupped= pd.merge(connector_view,data, on=['Connector Name','Pin Name'], how='inner')
len(connector_view),len(data),len(groupped)

In [ ]:
connector_view2=data.groupby(["Connector Name"]).agg({ 'Pin Name':['nunique','count'] })
connector_view2.head()

In [ ]:
levels=connector_view2.index.nlevels
for i in reversed(range(0,levels)):
    connector_view2.reset_index(level=i, inplace=True)

In [ ]:
connector_view2.columns=['Connector Name','Num_Pin_unique','Num_Pins']

In [ ]:
connector_view2['more_signal_in_pin']=connector_view2["Num_Pin_unique"]!=connector_view2["Num_Pins"]
connector_view2.head(25)

In [ ]:
groupped2= pd.merge(groupped,connector_view2, on=['Connector Name'], how='inner')
len(groupped2),len(groupped)

In [ ]:
groupped2.head(15)

In [ ]:
def category_signal(row):
    if row["Signal Name"][:5]=='BATT_':
        return 1
    elif row["Signal Name"][:4]=='CAN_':
        return 2
    elif row["Signal Name"][:6]=='CANFD_':
        return 3
    elif row["Signal Name"][:4]=='GND_':
        return 4
    elif row["Signal Name"][:9]=='GNDR_GND_':
        return 5
    else:
        return 6

In [ ]:
groupped2["signal_cat"]= groupped2.apply(lambda row:category_signal(row), axis=1 )

In [ ]:
def category_signal2(row):
    if row["Signal Name"][-3:]=='_HI' or row["Signal Name"][-3:]=='_LO':
        return 1
    else:
        return 0

In [ ]:
groupped2["signal_multicore"]= groupped2.apply(lambda row:category_signal2(row), axis=1 )

In [ ]:
groupped2[["Signal Name","signal_cat","signal_multicore"]].tail(25)

In [ ]:
groupped2['Cat_Signal Name'] = groupped2["Signal Name"].astype('category').cat.codes
c = groupped2["Signal Name"].astype('category')
d = dict(enumerate(c.cat.categories))

In [ ]:
def get_key(x,d):
    key_list=list(d.keys())
    val_list=list(d.values())
    ind=val_list.index(x["Pin PreferredSignal"])
    return key_list[ind]

In [ ]:
groupped2['Cat_PreferredSignal']= groupped2.apply(lambda row: get_key(row,d),axis=1)

In [ ]:
max_categories = groupped2['Cat_Signal Name'].max()

In [ ]:
connectors={}
conn_to_save=""
for i, row in groupped2.iterrows():
    connector=groupped2.loc[i,"Connector Name"]
    pin=groupped2.loc[i,'Pin Name']
    
    if conn_to_save=="" or groupped2.loc[i-1,"Connector Name"]==connector:
        try:
        
            connectors[connector][pin]={}
        except:
            connectors[connector]={}
            connectors[connector][pin]={}
            
        
        conn_to_save="NO"
    else:
        conn_to_save=""
        try:
        
            connectors[connector][pin]={}
        except:
            connectors[connector]={}
            connectors[connector][pin]={}
    
    #Part Number at level of PIN
    PartNumber =groupped2.loc[i,'Connector PartNumber']
    connectors[connector]["PartNumber"]=PartNumber
    #prefered Signal
    PreferredSignal=groupped2.loc[i,'Pin PreferredSignal']
    connectors[connector][pin]['PreferredSignal']=PreferredSignal
    Cat_PreferredSignal=groupped2.loc[i,'Cat_PreferredSignal']
    connectors[connector][pin]['Cat_PreferredSignal']=Cat_PreferredSignal
    
    # Signal in that pin
    Signal =groupped2.loc[i,'Signal Name']
    connectors[connector][pin]['Signal']=Signal
    Cat_Signal =groupped2.loc[i,'Cat_Signal Name']
    connectors[connector][pin]['Cat_Signal']=Cat_Signal
    #Check Signal 1 preferred = Signal, 0 1 preferred != Signal, 2 both NaN 
    check_signal=groupped2.loc[i,'check_signal']
    connectors[connector][pin]['check_signal']=check_signal
    #Number of pins connector
    Num_Pins =groupped2.loc[i,'Num_Pins']
    connectors[connector][pin]['Num_Pins']=Num_Pins
    #Number of unique Pins in this connector
    Num_Pin_unique=groupped2.loc[i,'Num_Pin_unique']
    connectors[connector][pin]['Num_Pin_unique']=Num_Pin_unique
    #True if number of Pins is not the same of number of unique Pins
    more_signal_in_pin =groupped2.loc[i,'more_signal_in_pin']
    connectors[connector][pin]['more_signal_in_pin']=more_signal_in_pin


In [ ]:
len(connectors['TO_111_LH_(121)'].keys())-1

In [ ]:
connectors['TO_111_LH_(121)'][1]['Cat_Signal']

In [ ]:
Cat_PreferredSignals=[]
for key in connectors['TO_111_LH_(121)'].keys():
    if key != 'PartNumber':
        Cat_PreferredSignals.append(connectors['TO_111_LH_(121)'][key]['Cat_PreferredSignal'])
        

In [ ]:
Cat_PreferredSignals

In [ ]:
import random

print(random.randint(0, max_categories ))

In [ ]:
for i in range(10):
    print(random.random())

In [ ]:
len(connectors.keys())

In [ ]:
len(data['Signal Name'].unique()),len(data['Pin PreferredSignal'].unique())

In [ ]:
signal_view=data.groupby(["Pin PreferredSignal"]).agg({"Pin Name":[ "count", 'nunique'], "Connector Name":'nunique' })
signal_view.head(10)

In [ ]:
groupped.to_csv("data/connectors.csv", index=False)

In [ ]:
groupped['Connector PartNumber'].fillna("NOPARTNUMBER", inplace=True)
groupped.head(20)